# 02. PaySim 계좌별 과거 거래 특징 생성

**목표:** 검증이 끝난 Train / Validation / Test Parquet을 PySpark로 다시 읽고, 거래가 발생한 `step`보다 앞선 계좌 거래만으로 과거 특징을 생성한다.

- 시간 단위인 `step`을 보존하고, 동일 `step`의 거래는 과거 이력에서 제외한다.
- 과거 거래 횟수·누적 금액·평균 금액·활동 step 수·마지막 거래 이후 간격을 송금인과 수취인별로 계산한다.
- 파생 이력 특징에는 사기 레이블을 포함하지 않는다. 현재 거래의 `isFraud`는 별도 평가 정답으로만 유지한다.
- 계좌 이름은 조사 조회에 유지하지만 모델 특징 허용 목록에는 넣지 않는다.
- 검증과 테스트 거래의 특징에는 더 이른 Train·Validation 거래가 과거 이력으로 포함된다.

프로젝트 `.venv`에서 위에서 아래로 실행한다. 전체 거래 데이터는 Spark DataFrame에서 처리하고 작은 요약만 노트북으로 가져온다.

## 1. 실행 설정과 검증된 분할 선택
완료된 실행의 `manifest.json`과 Parquet 산출물을 선택한다. 손상되거나 검증되지 않은 실행은 중단한다.

In [1]:
import json
import os
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "pyproject.toml").is_file() and (p / "src/fds_agent").is_dir()
    ),
    None,
)
assert ROOT is not None, "프로젝트 루트 또는 notebooks 폴더에서 실행한다."
SPLIT_ROOT = ROOT / "data/processed/paysim_time_split"
verified = []
for manifest_path in SPLIT_ROOT.glob("*/manifest.json"):
    candidate = json.loads(manifest_path.read_text(encoding="utf-8"))
    parquet_path = ROOT / candidate["parquet_path"]
    if candidate.get("verified") and parquet_path.is_dir():
        verified.append((manifest_path, candidate, parquet_path))
assert verified, "검증 완료된 시간순 Parquet 분할이 없다. 01 노트북을 먼저 실행한다."
source_manifest_path, source_manifest, INPUT_PATH = sorted(
    verified, key=lambda item: item[0].parent.name
)[-1]
assert source_manifest["overall"]["rows"] == 6_362_620
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
REPORT_DIR = ROOT / "artifacts/reports" / RUN_ID
OUTPUT_DIR = ROOT / "artifacts/features/paysim_temporal" / RUN_ID
REPORT_DIR.mkdir(parents=True, exist_ok=False)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
started = time.perf_counter()
print("Source split:", source_manifest_path.parent.name)
print("Input Parquet:", INPUT_PATH)
print("Feature run:", RUN_ID)
print("Source rows:", f"{source_manifest['overall']['rows']:,}")

Source split: 20261117T053742Z-09ddeaeb
Input Parquet: /Users/yueun/Desktop/FDS-AI-Agent/data/processed/paysim_time_split/20261117T053742Z-09ddeaeb/transactions
Feature run: 20261117T090124Z-c769f595
Source rows: 6,362,620


## 2. Spark executor와 입력 준비
두 worker의 local-cluster 모드에서 과거 거래 특징을 계산한다. 전체 데이터와 원본 분할 구간을 로드하고 label·거래 유형·step 범위 요약을 확인한다.

In [2]:
SPARK_WORKERS = 2
WORKER_CORES = 1
WORKER_MEMORY_MB = 1024
DRIVER_MEMORY = "1g"
EXECUTOR_MEMORY = "768m"
EVENT_LOG_DIR = REPORT_DIR / "spark-events"
EVENT_LOG_DIR.mkdir(parents=True, exist_ok=True)
if sys.platform == "darwin" and not os.environ.get("JAVA_HOME"):
    os.environ["JAVA_HOME"] = subprocess.check_output(
        ["/usr/libexec/java_home", "-v", "17"], text=True
    ).strip()
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
spark = (
    SparkSession.builder.master(f"local-cluster[{SPARK_WORKERS},{WORKER_CORES},{WORKER_MEMORY_MB}]")
    .appName("FDS-PaySim-Temporal-Features")
    .config("spark.driver.memory", DRIVER_MEMORY)
    .config("spark.executor.memory", EXECUTOR_MEMORY)
    .config("spark.executor.cores", str(WORKER_CORES))
    .config("spark.cores.max", str(SPARK_WORKERS * WORKER_CORES))
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", EVENT_LOG_DIR.as_uri())
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Spark UI:", spark.sparkContext.uiWebUrl)

df = spark.read.parquet(str(INPUT_PATH)).persist(StorageLevel.MEMORY_AND_DISK)
assert {"train", "validation", "test"}.issubset(
    set(df.select("split").distinct().toPandas()["split"])
)
assert "isFraud" in df.columns and "isFlaggedFraud" in df.columns
source_summary = (
    df.groupBy("split")
    .agg(
        F.count("*").alias("rows"),
        F.sum("isFraud").alias("fraud"),
        F.min("step").alias("min_step"),
        F.max("step").alias("max_step"),
    )
    .orderBy("min_step")
    .toPandas()
)
display(source_summary)
assert int(source_summary["rows"].sum()) == source_manifest["overall"]["rows"]

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 18:01:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.1
Spark master: local-cluster[2,1,1024]
Spark UI: http://localhost:4040


,split,rows,fraud,min_step,max_step
0,train,3820599,3193,1,281
1,validation,1248498,762,282,354
2,test,1293523,4258,355,743


## 3. 과거 step 집계 함수와 누출 경계 검증

먼저 작은 예제로 경계 규칙을 확인한다. `rangeBetween(unboundedPreceding, -1)`은 정렬값이 현재 step보다 작을 때만 포함한다. 같은 계좌의 같은 step에 송금 두 건이 있으면 두 건 모두 이력에서 제외한다.

In [3]:
def add_prior_step_features(step_events, prefix):
    history = (
        Window.partitionBy("account_id").orderBy("step").rangeBetween(Window.unboundedPreceding, -1)
    )
    return step_events.select(
        "account_id",
        "step",
        F.coalesce(F.sum("event_count").over(history), F.lit(0))
        .cast("long")
        .alias(f"{prefix}_prior_count"),
        F.coalesce(F.sum("event_amount_sum").over(history), F.lit(0.0)).alias(
            f"{prefix}_prior_amount_sum"
        ),
        F.coalesce(F.sum(F.lit(1)).over(history), F.lit(0))
        .cast("long")
        .alias(f"{prefix}_prior_active_steps"),
        F.max("step").over(history).alias(f"{prefix}_prior_last_step"),
    ).withColumn(
        f"{prefix}_prior_amount_mean",
        F.when(
            F.col(f"{prefix}_prior_count") > 0,
            F.col(f"{prefix}_prior_amount_sum") / F.col(f"{prefix}_prior_count"),
        ).otherwise(F.lit(0.0)),
    )


# 동일 step 안의 10·20 금액은 step 2에서 보이지 않는다. step 3에서는 35만 보인다.
toy_schema = T.StructType(
    [
        T.StructField("account_id", T.StringType(), False),
        T.StructField("step", T.IntegerType(), False),
        T.StructField("event_count", T.LongType(), False),
        T.StructField("event_amount_sum", T.DoubleType(), False),
    ]
)
toy = spark.createDataFrame([("A", 1, 2, 30.0), ("A", 2, 1, 5.0), ("A", 3, 1, 100.0)], toy_schema)
toy_features = add_prior_step_features(toy, "send").orderBy("step").collect()
assert toy_features[0]["send_prior_count"] == 0
assert toy_features[1]["send_prior_count"] == 2
assert toy_features[1]["send_prior_amount_sum"] == 30.0
assert toy_features[2]["send_prior_count"] == 3
assert toy_features[2]["send_prior_amount_sum"] == 35.0
assert toy_features[2]["send_prior_amount_mean"] == 35.0 / 3
print("경계 검증 통과: 동일 step과 미래 step을 이력에서 제외한다.")

경계 검증 통과: 동일 step과 미래 step을 이력에서 제외한다.


## 4. 송금인·수취인별 과거 이력 생성
각 계좌와 step 단위로 먼저 이벤트를 집계한다. 사기 여부는 집계 입력에 선택하지 않는다. 이후 계좌별 window를 적용하고 거래 테이블에 계좌·step 키로 one-to-one 결합한다.

In [4]:
event_source = df.select(
    F.col("nameOrig").alias("account_id"), "step", F.col("amount").cast("double").alias("amount")
)
event_destination = df.select(
    F.col("nameDest").alias("account_id"), "step", F.col("amount").cast("double").alias("amount")
)


def aggregate_account_steps(events):
    return events.groupBy("account_id", "step").agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("event_amount_sum"),
    )


send_history = (
    add_prior_step_features(aggregate_account_steps(event_source), "send")
    .withColumnRenamed("account_id", "_send_account_id")
    .withColumnRenamed("step", "_send_step")
)
receive_history = (
    add_prior_step_features(aggregate_account_steps(event_destination), "receive")
    .withColumnRenamed("account_id", "_receive_account_id")
    .withColumnRenamed("step", "_receive_step")
)
features = (
    df.join(
        send_history,
        (df["nameOrig"] == send_history["_send_account_id"])
        & (df["step"] == send_history["_send_step"]),
        "left",
    )
    .drop("_send_account_id", "_send_step")
    .join(
        receive_history,
        (df["nameDest"] == receive_history["_receive_account_id"])
        & (df["step"] == receive_history["_receive_step"]),
        "left",
    )
    .drop("_receive_account_id", "_receive_step")
)
for role in ["send", "receive"]:
    for metric in ["prior_count", "prior_amount_sum", "prior_active_steps", "prior_amount_mean"]:
        column = f"{role}_{metric}"
        features = features.withColumn(column, F.coalesce(F.col(column), F.lit(0)))
    gap = F.col("step") - F.col(f"{role}_prior_last_step")
    features = features.withColumn(
        f"{role}_step_gap", F.when(gap.isNull(), F.lit(-1)).otherwise(gap).cast("int")
    ).drop(f"{role}_prior_last_step")

features = features.withColumn("amount_log1p", F.log1p(F.col("amount")))
features = features.withColumn("oldbalanceOrg_log1p", F.log1p(F.col("oldbalanceOrg")))
features = features.withColumn("oldbalanceDest_log1p", F.log1p(F.col("oldbalanceDest")))
MODEL_FEATURES = [
    "step",
    "type",
    "amount",
    "amount_log1p",
    "oldbalanceOrg",
    "oldbalanceOrg_log1p",
    "oldbalanceDest",
    "oldbalanceDest_log1p",
    "send_prior_count",
    "send_prior_amount_sum",
    "send_prior_amount_mean",
    "send_prior_active_steps",
    "send_step_gap",
    "receive_prior_count",
    "receive_prior_amount_sum",
    "receive_prior_amount_mean",
    "receive_prior_active_steps",
    "receive_step_gap",
]
IDENTIFIER_COLUMNS = ["nameOrig", "nameDest", "split"]
TARGET_COLUMN = "isFraud"
assert TARGET_COLUMN not in MODEL_FEATURES
assert "isFlaggedFraud" not in MODEL_FEATURES
assert "newbalanceOrig" not in MODEL_FEATURES and "newbalanceDest" not in MODEL_FEATURES
feature_columns = IDENTIFIER_COLUMNS + MODEL_FEATURES + [TARGET_COLUMN, "isFlaggedFraud"]
features = features.select(*feature_columns).persist(StorageLevel.MEMORY_AND_DISK)
print("Model feature count:", len(MODEL_FEATURES))
print("History fields:", [c for c in MODEL_FEATURES if c.startswith(("send_", "receive_"))])
print("History inputs include labels:", False)

Model feature count: 18
History fields: ['send_prior_count', 'send_prior_amount_sum', 'send_prior_amount_mean', 'send_prior_active_steps', 'send_step_gap', 'receive_prior_count', 'receive_prior_amount_sum', 'receive_prior_amount_mean', 'receive_prior_active_steps', 'receive_step_gap']
History inputs include labels: False


## 5. 특징 품질·시점 경계 확인
분할별 거래 건수와 정답 건수를 원본에 대조한다. 특징 결측, 계좌·step 중복, 잘못된 분할 경계를 검사한다. 계좌의 동일 step에는 history join 키가 한 행뿐이므로 거래 중복을 늘리지 않는다.

In [5]:
feature_summary = (
    features.groupBy("split")
    .agg(
        F.count("*").alias("rows"),
        F.sum(TARGET_COLUMN).alias("fraud"),
        F.sum(F.when(F.col("send_prior_count").isNull(), 1).otherwise(0)).alias("null_send_count"),
        F.sum(F.when(F.col("receive_prior_count").isNull(), 1).otherwise(0)).alias(
            "null_receive_count"
        ),
        F.min("step").alias("min_step"),
        F.max("step").alias("max_step"),
    )
    .orderBy("min_step")
    .toPandas()
)
assert feature_summary["rows"].tolist() == source_summary["rows"].tolist()
assert feature_summary["fraud"].tolist() == source_summary["fraud"].tolist()
assert int(feature_summary["null_send_count"].sum()) == 0
assert int(feature_summary["null_receive_count"].sum()) == 0
assert features.filter(F.col("amount_log1p").isNull()).limit(1).count() == 0
assert features.filter(F.col("oldbalanceOrg_log1p").isNull()).limit(1).count() == 0
assert features.filter(F.col("oldbalanceDest_log1p").isNull()).limit(1).count() == 0
display(feature_summary)

profile = features.agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("send_prior_count") == 0, 1).otherwise(0)).alias("no_prior_send"),
    F.sum(F.when(F.col("receive_prior_count") == 0, 1).otherwise(0)).alias("no_prior_receive"),
    F.max("send_prior_count").alias("max_prior_send_count"),
    F.max("receive_prior_count").alias("max_prior_receive_count"),
).toPandas()
display(profile)

,split,rows,fraud,null_send_count,null_receive_count,min_step,max_step
0,train,3820599,3193,0,0,1,281
1,validation,1248498,762,0,0,282,354
2,test,1293523,4258,0,0,355,743


,rows,no_prior_send,no_prior_receive,max_prior_send_count,max_prior_receive_count
0,6362620,6353354,2769903,2,112


## 6. Parquet 저장과 산출물 재검증
실행별 경로를 사용해 기존 feature 데이터를 덮어쓰지 않는다. 분할별 label·행 수·step 범위를 다시 비교한다. 모델은 manifest의 `model_features` 허용 목록만 입력으로 사용한다.

In [6]:
FEATURE_PATH = OUTPUT_DIR / "transactions"
write_started = time.perf_counter()
(
    features.repartition(8)
    .write.mode("errorifexists")
    .partitionBy("split")
    .parquet(str(FEATURE_PATH))
)
restored = spark.read.parquet(str(FEATURE_PATH))
restored_summary = (
    restored.groupBy("split")
    .agg(
        F.count("*").alias("rows"),
        F.sum(TARGET_COLUMN).alias("fraud"),
        F.min("step").alias("min_step"),
        F.max("step").alias("max_step"),
    )
    .orderBy("min_step")
    .toPandas()
)
pd.testing.assert_frame_equal(
    restored_summary,
    feature_summary[["split", "rows", "fraud", "min_step", "max_step"]],
    check_dtype=False,
)
assert restored.select("split").distinct().count() == 3
write_seconds = time.perf_counter() - write_started
print("Parquet read-back verification passed.")
print("Feature parquet:", FEATURE_PATH)
print("Write and read-back seconds:", round(write_seconds, 2))

Parquet read-back verification passed.
Feature parquet: /Users/yueun/Desktop/FDS-AI-Agent/artifacts/features/paysim_temporal/20261117T090124Z-c769f595/transactions
Write and read-back seconds: 16.03


## 7. Spark UI 처리 지표와 실행 manifest
Spark UI 지표를 저장해 worker 분담과 shuffle 양을 기록한다. 완료한 application의 UI 지표를 읽고 실행 결과 manifest를 작성한다.

In [7]:
import urllib.request


def read_spark_ui(endpoint):
    url = f"{spark.sparkContext.uiWebUrl}/api/v1/{endpoint}"
    with urllib.request.urlopen(url, timeout=10) as response:
        return json.loads(response.read().decode("utf-8"))


app_id = spark.sparkContext.applicationId
applications = read_spark_ui("applications")
app_match = next((a for a in applications if a["id"] == app_id), None)
assert app_match is not None
api_root = f"applications/{app_match['id']}"
jobs = read_spark_ui(f"{api_root}/jobs?status=succeeded")
stages = read_spark_ui(f"{api_root}/stages?status=complete&withSummaries=true&quantiles=0.5")
executors = read_spark_ui(f"{api_root}/allexecutors")
workers = [e for e in executors if str(e.get("id")) != "driver"]
assert len(workers) == SPARK_WORKERS
assert sum(e.get("completedTasks", 0) for e in workers) > 0
executor_metrics = [
    {
        key: executor.get(key)
        for key in [
            "id",
            "hostPort",
            "totalCores",
            "completedTasks",
            "failedTasks",
            "totalInputBytes",
            "totalShuffleRead",
            "totalShuffleWrite",
        ]
    }
    for executor in executors
]
print("Spark jobs:", len(jobs), "| stages:", len(stages))
display(
    pd.DataFrame(workers)[
        [
            "id",
            "hostPort",
            "totalCores",
            "completedTasks",
            "failedTasks",
            "totalInputBytes",
            "totalShuffleRead",
            "totalShuffleWrite",
        ]
    ]
)
metrics = {
    "run_id": RUN_ID,
    "application_id": app_id,
    "master": spark.sparkContext.master,
    "ui_url": spark.sparkContext.uiWebUrl,
    "worker_count": SPARK_WORKERS,
    "jobs": jobs,
    "stages": stages,
    "executors": executor_metrics,
    "event_log_dir": EVENT_LOG_DIR.as_uri(),
}
(REPORT_DIR / "spark_ui_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
source_info = {
    "source_split_manifest": str(source_manifest_path.relative_to(ROOT)),
    "source_split_run_id": source_manifest["run_id"],
    "input_rows": source_manifest["overall"]["rows"],
}
manifest = {
    "run_id": RUN_ID,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": source_info,
    "spark": spark.version,
    "spark_master": spark.sparkContext.master,
    "rows": int(feature_summary["rows"].sum()),
    "splits": json.loads(feature_summary.to_json(orient="records")),
    "history_policy": {
        "window": "partition by account_id order by step range between unbounded preceding and -1",
        "same_step_history_allowed": False,
        "future_step_history_allowed": False,
        "history_aggregates_include_labels": False,
    },
    "model_features": MODEL_FEATURES,
    "identifiers_retained_for_investigation": ["nameOrig", "nameDest"],
    "label": TARGET_COLUMN,
    "excluded_from_model_features": [
        "isFlaggedFraud",
        "newbalanceOrig",
        "newbalanceDest",
        "nameOrig",
        "nameDest",
    ],
    "parquet_path": str(FEATURE_PATH.relative_to(ROOT)),
    "report_path": str(REPORT_DIR.relative_to(ROOT)),
    "verified": True,
    "verification": "split row/label counts match input; Parquet read-back summary matches",
    "spark_jobs": len(jobs),
    "spark_stages": len(stages),
    "spark_workers": len(workers),
    "spark_completed_executor_tasks": int(sum(e.get("completedTasks", 0) for e in workers)),
    "spark_failed_executor_tasks": int(sum(e.get("failedTasks", 0) for e in workers)),
    "elapsed_seconds": round(time.perf_counter() - started, 2),
    "write_and_verify_seconds": round(write_seconds, 2),
}
text = json.dumps(manifest, ensure_ascii=False, indent=2)
(OUTPUT_DIR / "manifest.json").write_text(text, encoding="utf-8")
(REPORT_DIR / "manifest.json").write_text(text, encoding="utf-8")
print("Completed:", manifest["rows"], "rows |", manifest["elapsed_seconds"], "seconds")
print("Manifest:", OUTPUT_DIR / "manifest.json")
features.unpersist()
df.unpersist()
spark.stop()

Spark jobs: 43 | stages: 43


,id,hostPort,totalCores,completedTasks,failedTasks,totalInputBytes,totalShuffleRead,totalShuffleWrite
0,0,127.0.0.1:59806,1,66,0,1907998863,1079640642,1036918709
1,1,127.0.0.1:59808,1,68,0,2185989847,1115562735,1158281725


Completed: 6362620 rows | 72.16 seconds
Manifest: /Users/yueun/Desktop/FDS-AI-Agent/artifacts/features/paysim_temporal/20261117T090124Z-c769f595/manifest.json


## 다음 단계

1. `manifest.json`의 `model_features` allowlist로 LightGBM 학습 데이터를 구성한다.
2. Train으로 학습하고 Validation으로 threshold와 하이퍼파라미터를 선택한다.
3. Test 평가는 단 한 번 수행하고 테스트 정답은 거래 조사 Agent에 전달하지 않는다.
4. Agent에는 `nameOrig`, `nameDest` 조회 키와 과거 거래 기반 근거만 별도 전달한다.